# SefariaExport על Kaggle

- הפעל Internet למחברת.
- הרץ את תאי הקוד מלמעלה למטה.
- Kaggle או Papermill עלולים להפיל תא בודד שרץ יותר מ-12 שעות, לכן ה-pipeline פוצל לכמה תאים.
- אם תא נכשל, בדרך כלל אפשר לתקן את הבעיה ולהריץ שוב רק מהתא שנכשל ואילך.
- כברירת מחדל המחברת מקבעת את `SefariaExport` ל-commit `be22bd00bee3cbe67fb2b99e7af88d162435a0e1`.
- MongoDB מותקן מבינרי Community רשמיים בגרסה `6.0.27`, תואם ל-Compose.
- כל הקבצים הזמניים נשמרים תחת `/tmp`.
- רק התוצרים הסופיים מועתקים ל-`/kaggle/working`.
- העלאת Release ל-GitHub אופציונלית ומכובה כברירת מחדל.
- פרסום עדכון ל"פורום אוצריא" אופציונלי ומחייב `ENABLE_GITHUB_RELEASE=true` יחד עם secrets בשם `USER_NAME` ו-`PASSWORD`.


In [ ]:
%%bash
set -euo pipefail

BASE="/tmp/sefariaexport_run"
OUT_DIR="$BASE/final-artifacts"
PUBLISH_DIR="/kaggle/working"
CONFIG="$BASE/notebook.env"
LOG_DIR="$BASE/logs"
STATE_DIR="$BASE/state"

rm -rf "$BASE"
mkdir -p "$BASE" "$OUT_DIR" "$PUBLISH_DIR" "$LOG_DIR" "$STATE_DIR"

cat > "$CONFIG" <<'EOF'
export BASE=/tmp/sefariaexport_run
export OUT_DIR=/tmp/sefariaexport_run/final-artifacts
export PUBLISH_DIR=/kaggle/working
export CONFIG=/tmp/sefariaexport_run/notebook.env
export LOG_DIR=/tmp/sefariaexport_run/logs
export STATE_DIR=/tmp/sefariaexport_run/state
export TZ_NAME=Asia/Jerusalem
export PYTHON_VERSION=3.9
export MONGO_VERSION=6.0.27
export MONGO_TOOLS_VERSION=100.9.4
export SEFARIAEXPORT_REF=be22bd00bee3cbe67fb2b99e7af88d162435a0e1
export DUMP_TGZ_PATH=
export ENABLE_GITHUB_RELEASE=false
export ENABLE_OTZARIA_FORUM_POST=false
export GH_REPO=
EOF

printf 'Configured: %s\n' "$CONFIG"
printf 'Temporary artifacts dir: %s\n' "$OUT_DIR"
printf 'Published artifacts dir: %s\n' "$PUBLISH_DIR"
printf 'Edit %s if you want a custom dump, GitHub release upload, or Otzaria Forum update.\n' "$CONFIG"


In [ ]:
%%bash
set -euo pipefail
source /tmp/sefariaexport_run/notebook.env

timestamp() { date '+%H:%M:%S'; }
log() { printf '\n[%s] %s\n' "$(timestamp)" "$1"; }
run_step() {
  local name="$1"
  shift
  local log_file="$LOG_DIR/${name}.log"
  if "$@" >"$log_file" 2>&1; then
    printf '  ok\n'
  else
    printf '  failed, tail of %s:\n' "$log_file"
    tail -n 80 "$log_file"
    return 1
  fi
}

BIN_DIR="/tmp/bin"
MM_ROOT="/tmp/micromamba"
MM="$BIN_DIR/micromamba"
mkdir -p "$BIN_DIR" "$MM_ROOT" "$LOG_DIR" "$BASE/bin" "$BASE/mongo-data" "$BASE/mongo-logs" "$OUT_DIR" "$STATE_DIR"

if [ ! -x "$MM" ]; then
  log "Installing micromamba"
  curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C "$BIN_DIR" --strip-components=1 bin/micromamba
fi

if ! "$MM" -r "$MM_ROOT" env list | awk '{print $1}' | grep -qx 'sefariaexport'; then
  log "Creating Python ${PYTHON_VERSION} environment"
  run_step "create_python_env" "$MM" -r "$MM_ROOT" create -y -n sefariaexport -c conda-forge "python=${PYTHON_VERSION}" pip git curl wget tar unzip jq
fi

log "Updating pip"
run_step "upgrade_pip" "$MM" -r "$MM_ROOT" run -n sefariaexport pip install -U pip setuptools wheel

log "Cloning pinned SefariaExport"
cd "$BASE"
if [ ! -d "$BASE/SefariaExport/.git" ]; then
  run_step "clone_sefariaexport" git clone https://github.com/Otzaria/SefariaExport.git
else
  printf '  ok\n'
fi
cd "$BASE/SefariaExport"
run_step "checkout_sefariaexport_ref" git checkout --detach "$SEFARIAEXPORT_REF"

export GITHUB_WORKSPACE="$PWD"
export MONGO_HOST=127.0.0.1
export MONGO_PORT=27017
export MONGO_DB_NAME=sefaria
export RESTORE_INDEXES_MODE=skip_links
export SEFARIA_EXPORT_PATH="$GITHUB_WORKSPACE/exports"
export PATH="$BASE/bin:$PATH"
export PIP_NO_CACHE_DIR=1
export PIP_DISABLE_PIP_VERSION_CHECK=1
export PIP_ROOT_USER_ACTION=ignore

log "Computing timestamp"
if ! source ./01_compute_timestamp.sh >"$LOG_DIR/01_compute_timestamp.log" 2>&1; then
  tail -n 80 "$LOG_DIR/01_compute_timestamp.log"
  exit 1
fi
printf '  %s\n' "$TS_STAMP"

log "Installing base tools"
run_step "02_install_base_tools" bash 02_install_base_tools.sh

log "Installing MongoDB Database Tools"
run_step "install_mongo_tools" bash -lc '
  TGZ="mongodb-database-tools-ubuntu2204-x86_64-${MONGO_TOOLS_VERSION}.tgz"
  wget -q -O "$BASE/$TGZ" "https://fastdl.mongodb.org/tools/db/$TGZ"
  tar -xzf "$BASE/$TGZ" -C "$BASE"
  cp -a "$BASE/mongodb-database-tools-ubuntu2204-x86_64-${MONGO_TOOLS_VERSION}/bin/"* "$BASE/bin/"
  rm -rf "$BASE/mongodb-database-tools-ubuntu2204-x86_64-${MONGO_TOOLS_VERSION}" "$BASE/$TGZ"
'

log "Installing MongoDB ${MONGO_VERSION}"
run_step "install_mongod" bash -lc '
  TGZ="mongodb-linux-x86_64-ubuntu2204-${MONGO_VERSION}.tgz"
  curl -fsSL -o "$BASE/$TGZ" "https://fastdl.mongodb.org/linux/$TGZ"
  curl -fsSL -o "$BASE/$TGZ.sha256" "https://fastdl.mongodb.org/linux/$TGZ.sha256"
  (cd "$BASE" && sha256sum -c "$(basename "$TGZ").sha256")
  rm -rf "$BASE/mongo-server"
  mkdir -p "$BASE/mongo-server"
  tar -xzf "$BASE/$TGZ" -C "$BASE/mongo-server" --strip-components=1
  rm -f "$BASE/$TGZ" "$BASE/$TGZ.sha256"
'

if pgrep -f "mongod.*$BASE/mongo-data" >/dev/null 2>&1; then
  log "Stopping previous MongoDB instance"
  pkill -f "mongod.*$BASE/mongo-data" || true
  sleep 2
fi

log "Starting MongoDB"
run_step "start_mongod" bash -lc '
  rm -rf "$BASE/mongo-data"
  mkdir -p "$BASE/mongo-data" "$BASE/mongo-logs"
  nohup "$BASE/mongo-server/bin/mongod" \
    --dbpath "$BASE/mongo-data" \
    --bind_ip 127.0.0.1 \
    --port 27017 \
    --wiredTigerCacheSizeGB 2.5 \
    --logpath "$BASE/mongo-logs/mongod.log" \
    --logappend \
    > "$BASE/mongo-logs/nohup.out" 2>&1 &
'
run_step "11_wait_for_mongodb" bash 11_wait_for_mongodb.sh

if [ -n "$DUMP_TGZ_PATH" ]; then
  log "Preparing custom dump"
  run_step "prepare_custom_dump" bash -lc '
    rm -rf mongo_dump_pkg mongo_dump dump_small.tar.gz
    mkdir -p mongo_dump_pkg mongo_dump
    cp -a "$DUMP_TGZ_PATH" dump_small.tar.gz
    tar -xzf dump_small.tar.gz -C mongo_dump
    rm -f dump_small.tar.gz
    if [ -d mongo_dump/dump/sefaria ]; then
      cp -a mongo_dump/dump/sefaria mongo_dump_pkg/
    elif [ -d mongo_dump/sefaria ]; then
      cp -a mongo_dump/sefaria mongo_dump_pkg/
    else
      echo "sefaria folder not found in dump"
      exit 1
    fi
    rm -rf mongo_dump
  '
else
  log "Downloading default dump"
  run_step "04_download_small_dump" bash 04_download_small_dump.sh
fi

log "Cloning Sefaria-Project"
run_step "05_clone_sefaria_project" "$MM" -r "$MM_ROOT" run -n sefariaexport bash 05_clone_sefaria_project.sh

log "Installing build dependencies"
if ! run_step "06_install_build_deps" bash 06_install_build_deps.sh; then
  printf '  continuing without extra build deps\n'
fi

log "Installing Python requirements"
if ! run_step "07_pip_install_requirements" "$MM" -r "$MM_ROOT" run -n sefariaexport bash 07_pip_install_requirements.sh; then
  log "Using google-re2 fallback"
  run_step "08_fallback_built_google_re2" "$MM" -r "$MM_ROOT" run -n sefariaexport bash 08_fallback_built_google_re2.sh
fi

rm -rf "$SEFARIA_EXPORT_PATH"
mkdir -p "$SEFARIA_EXPORT_PATH"

log "Preparing export settings"
run_step "09_create_exports_dir" "$MM" -r "$MM_ROOT" run -n sefariaexport bash 09_create_exports_dir.sh
run_step "10_create_local_settings" "$MM" -r "$MM_ROOT" run -n sefariaexport bash 10_create_local_settings.sh

log "Restoring MongoDB dump"
run_step "12_restore_db_from_dump" "$MM" -r "$MM_ROOT" run -n sefariaexport bash 12_restore_db_from_dump.sh

log "Preparation complete"
printf '  MongoDB is still running for the next cells.\n'
printf '  Continue with the export cells below.\n'


In [ ]:
%%bash
set -euo pipefail
source /tmp/sefariaexport_run/notebook.env

timestamp() { date '+%H:%M:%S'; }
log() { printf '\n[%s] %s\n' "$(timestamp)" "$1"; }
run_step() {
  local name="$1"
  shift
  local log_file="$LOG_DIR/${name}.log"
  if "$@" >"$log_file" 2>&1; then
    printf '  ok\n'
  else
    printf '  failed, tail of %s:\n' "$log_file"
    tail -n 80 "$log_file"
    return 1
  fi
}

BIN_DIR="/tmp/bin"
MM_ROOT="/tmp/micromamba"
MM="$BIN_DIR/micromamba"
REPO_DIR="$BASE/SefariaExport"
cd "$REPO_DIR"

export GITHUB_WORKSPACE="$REPO_DIR"
export MONGO_HOST=127.0.0.1
export MONGO_PORT=27017
export MONGO_DB_NAME=sefaria
export RESTORE_INDEXES_MODE=skip_links
export SEFARIA_EXPORT_PATH="$GITHUB_WORKSPACE/exports"
export PATH="$BASE/bin:$PATH"
export PIP_NO_CACHE_DIR=1
export PIP_DISABLE_PIP_VERSION_CHECK=1
export PIP_ROOT_USER_ACTION=ignore

log "Checking exporter module"
run_step "13_check_export_module" "$MM" -r "$MM_ROOT" run -n sefariaexport bash 13_check_export_module.sh

log "Running export_all_merged"
run_step "14_run_export_all_merged" "$MM" -r "$MM_ROOT" run -n sefariaexport python run_exports.py export_all_merged

printf '%s\n' export_all_merged > "$STATE_DIR/last-successful-export-step.txt"
log "export_all_merged complete"
printf '  You can continue to the next export cell.\n'


In [ ]:
%%bash
set -euo pipefail
source /tmp/sefariaexport_run/notebook.env

timestamp() { date '+%H:%M:%S'; }
log() { printf '\n[%s] %s\n' "$(timestamp)" "$1"; }
run_step() {
  local name="$1"
  shift
  local log_file="$LOG_DIR/${name}.log"
  if "$@" >"$log_file" 2>&1; then
    printf '  ok\n'
  else
    printf '  failed, tail of %s:\n' "$log_file"
    tail -n 80 "$log_file"
    return 1
  fi
}

BIN_DIR="/tmp/bin"
MM_ROOT="/tmp/micromamba"
MM="$BIN_DIR/micromamba"
REPO_DIR="$BASE/SefariaExport"
cd "$REPO_DIR"

export GITHUB_WORKSPACE="$REPO_DIR"
export MONGO_HOST=127.0.0.1
export MONGO_PORT=27017
export MONGO_DB_NAME=sefaria
export RESTORE_INDEXES_MODE=skip_links
export SEFARIA_EXPORT_PATH="$GITHUB_WORKSPACE/exports"
export PATH="$BASE/bin:$PATH"
export PIP_NO_CACHE_DIR=1
export PIP_DISABLE_PIP_VERSION_CHECK=1
export PIP_ROOT_USER_ACTION=ignore

log "Running export_links"
run_step "14_run_export_links" "$MM" -r "$MM_ROOT" run -n sefariaexport python run_exports.py export_links

printf '%s\n' export_links > "$STATE_DIR/last-successful-export-step.txt"
log "export_links complete"
printf '  Continue to the next export cell.\n'


In [ ]:
%%bash
set -euo pipefail
source /tmp/sefariaexport_run/notebook.env

timestamp() { date '+%H:%M:%S'; }
log() { printf '\n[%s] %s\n' "$(timestamp)" "$1"; }
run_step() {
  local name="$1"
  shift
  local log_file="$LOG_DIR/${name}.log"
  if "$@" >"$log_file" 2>&1; then
    printf '  ok\n'
  else
    printf '  failed, tail of %s:\n' "$log_file"
    tail -n 80 "$log_file"
    return 1
  fi
}

BIN_DIR="/tmp/bin"
MM_ROOT="/tmp/micromamba"
MM="$BIN_DIR/micromamba"
REPO_DIR="$BASE/SefariaExport"
cd "$REPO_DIR"

export GITHUB_WORKSPACE="$REPO_DIR"
export MONGO_HOST=127.0.0.1
export MONGO_PORT=27017
export MONGO_DB_NAME=sefaria
export RESTORE_INDEXES_MODE=skip_links
export SEFARIA_EXPORT_PATH="$GITHUB_WORKSPACE/exports"
export PATH="$BASE/bin:$PATH"
export PIP_NO_CACHE_DIR=1
export PIP_DISABLE_PIP_VERSION_CHECK=1
export PIP_ROOT_USER_ACTION=ignore

log "Running export_schemas"
run_step "14_run_export_schemas" "$MM" -r "$MM_ROOT" run -n sefariaexport python run_exports.py export_schemas

printf '%s\n' export_schemas > "$STATE_DIR/last-successful-export-step.txt"
log "export_schemas complete"
printf '  Continue to the next export cell.\n'


In [ ]:
%%bash
set -euo pipefail
source /tmp/sefariaexport_run/notebook.env

timestamp() { date '+%H:%M:%S'; }
log() { printf '\n[%s] %s\n' "$(timestamp)" "$1"; }
run_step() {
  local name="$1"
  shift
  local log_file="$LOG_DIR/${name}.log"
  if "$@" >"$log_file" 2>&1; then
    printf '  ok\n'
  else
    printf '  failed, tail of %s:\n' "$log_file"
    tail -n 80 "$log_file"
    return 1
  fi
}

BIN_DIR="/tmp/bin"
MM_ROOT="/tmp/micromamba"
MM="$BIN_DIR/micromamba"
REPO_DIR="$BASE/SefariaExport"
cd "$REPO_DIR"

export GITHUB_WORKSPACE="$REPO_DIR"
export MONGO_HOST=127.0.0.1
export MONGO_PORT=27017
export MONGO_DB_NAME=sefaria
export RESTORE_INDEXES_MODE=skip_links
export SEFARIA_EXPORT_PATH="$GITHUB_WORKSPACE/exports"
export PATH="$BASE/bin:$PATH"
export PIP_NO_CACHE_DIR=1
export PIP_DISABLE_PIP_VERSION_CHECK=1
export PIP_ROOT_USER_ACTION=ignore

log "Running export_toc"
run_step "14_run_export_toc" "$MM" -r "$MM_ROOT" run -n sefariaexport python run_exports.py export_toc

printf '%s\n' export_toc > "$STATE_DIR/last-successful-export-step.txt"
log "export_toc complete"
printf '  Continue to the packaging cell.\n'


In [ ]:
%%bash
set -euo pipefail
source /tmp/sefariaexport_run/notebook.env

timestamp() { date '+%H:%M:%S'; }
log() { printf '\n[%s] %s\n' "$(timestamp)" "$1"; }
run_step() {
  local name="$1"
  shift
  local log_file="$LOG_DIR/${name}.log"
  if "$@" >"$log_file" 2>&1; then
    printf '  ok\n'
  else
    printf '  failed, tail of %s:\n' "$log_file"
    tail -n 80 "$log_file"
    return 1
  fi
}
cleanup() {
  if pgrep -f "mongod.*$BASE/mongo-data" >/dev/null 2>&1; then
    pkill -f "mongod.*$BASE/mongo-data" || true
  fi
}
trap cleanup EXIT

BIN_DIR="/tmp/bin"
MM_ROOT="/tmp/micromamba"
MM="$BIN_DIR/micromamba"
REPO_DIR="$BASE/SefariaExport"
cd "$REPO_DIR"

export GITHUB_WORKSPACE="$REPO_DIR"
export MONGO_HOST=127.0.0.1
export MONGO_PORT=27017
export MONGO_DB_NAME=sefaria
export RESTORE_INDEXES_MODE=skip_links
export SEFARIA_EXPORT_PATH="$GITHUB_WORKSPACE/exports"
export PATH="$BASE/bin:$PATH"
export PIP_NO_CACHE_DIR=1
export PIP_DISABLE_PIP_VERSION_CHECK=1
export PIP_ROOT_USER_ACTION=ignore

log "Verifying exports"
run_step "15_verify_exports" "$MM" -r "$MM_ROOT" run -n sefariaexport bash 15_verify_exports.sh
if ! run_step "16_drop_db" bash 16_drop_db.sh; then
  printf '  continuing after non-blocking drop-db step\n'
fi

log "Post-processing and archiving"
run_step "17a_remove_english_in_exports" bash 17a_remove_english_in_exports.sh
run_step "17b_flatten_hebrew_in_exports" bash 17b_flatten_hebrew_in_exports.sh
run_step "17_build_combined_archive" bash 17_build_combined_archive.sh
run_step "18_split_archive" bash 18_split_archive.sh

shopt -s nullglob
ARTIFACTS=( "sefaria-exports-${TS_STAMP}.tar.zst" "sefaria-exports-${TS_STAMP}.tar.zst.part-"* )
FINAL_ARTIFACTS=()
for f in "${ARTIFACTS[@]}"; do
  if [ -e "$f" ]; then
    FINAL_ARTIFACTS+=("$f")
  fi
done
if [ "${#FINAL_ARTIFACTS[@]}" -eq 0 ]; then
  echo "❌ no archive files were created"
  exit 1
fi
rm -f "$OUT_DIR"/sefaria-exports-"$TS_STAMP"* "$PUBLISH_DIR"/sefaria-exports-"$TS_STAMP"*
cp -f "${FINAL_ARTIFACTS[@]}" "$OUT_DIR/"
(cd "$OUT_DIR" && sha256sum "${FINAL_ARTIFACTS[@]##*/}" > "sefaria-exports-${TS_STAMP}.sha256")
printf '%s\n' "$TS_STAMP" > "$STATE_DIR/latest-release.stamp.txt"
cp -f "$OUT_DIR"/sefaria-exports-"$TS_STAMP"* "$PUBLISH_DIR/"

if [ "$ENABLE_GITHUB_RELEASE" = "true" ]; then
  log "Publishing GitHub release"
  : "${GH_REPO:?Set GH_REPO in $CONFIG when ENABLE_GITHUB_RELEASE=true}"
  export GH_REPO
  export GH_TOKEN="$(python3 -c 'from kaggle_secrets import UserSecretsClient; print(UserSecretsClient().get_secret("GH_TOKEN"))')"
  run_step "19_ensure_gh_cli" bash 19_ensure_gh_cli.sh
  run_step "20_create_or_update_release" bash 20_create_or_update_release.sh
  run_step "21_upload_release_assets" bash 21_upload_release_assets.sh
fi

log "Done"
printf '  TS_STAMP=%s\n' "$TS_STAMP"
printf '  temp artifacts: %s\n' "$OUT_DIR"
ls -lh "$PUBLISH_DIR"/sefaria-exports-"$TS_STAMP"*
printf '  logs: %s\n' "$LOG_DIR"
printf '  MongoDB has been stopped at the end of this cell.\n'


In [ ]:
%%bash
set -euo pipefail
source /tmp/sefariaexport_run/notebook.env

timestamp() { date '+%H:%M:%S'; }
log() { printf '\n[%s] %s\n' "$(timestamp)" "$1"; }
run_step() {
  local name="$1"
  shift
  local log_file="$LOG_DIR/${name}.log"
  if "$@" >"$log_file" 2>&1; then
    printf '  ok\n'
  else
    printf '  failed, tail of %s:\n' "$log_file"
    tail -n 80 "$log_file"
    return 1
  fi
}
get_secret() {
  python3 - "$1" <<'PY'
import sys

name = sys.argv[1]
value = ""
try:
    from kaggle_secrets import UserSecretsClient
    value = UserSecretsClient().get_secret(name) or ""
except Exception:
    value = ""
print(value)
PY
}

CONFIG="$BASE/notebook.env"
REPO_DIR="$BASE/SefariaExport"

if [ "$ENABLE_OTZARIA_FORUM_POST" != "true" ]; then
  log "Skipping Otzaria Forum update"
  printf '  set ENABLE_OTZARIA_FORUM_POST=true in %s to enable this step\n' "$CONFIG"
  exit 0
fi

if [ "$ENABLE_GITHUB_RELEASE" != "true" ]; then
  log "Skipping Otzaria Forum update"
  printf '  this step requires ENABLE_GITHUB_RELEASE=true because it compares against the previous GitHub release\n'
  exit 0
fi

if [ -z "$GH_REPO" ]; then
  echo "Set GH_REPO in $CONFIG when ENABLE_OTZARIA_FORUM_POST=true"
  exit 1
fi

STAMP_FILE="$STATE_DIR/latest-release.stamp.txt"
if [ ! -f "$STAMP_FILE" ]; then
  echo "No stamp file found under $STATE_DIR"
  exit 1
fi
TS_STAMP=$(cat "$STAMP_FILE")

if ! command -v gh >/dev/null 2>&1; then
  log "Installing GitHub CLI"
  run_step "19_ensure_gh_cli_for_forum" bash "$REPO_DIR/19_ensure_gh_cli.sh"
fi

GH_TOKEN=$(get_secret GH_TOKEN)
USER_NAME=$(get_secret USER_NAME)
PASSWORD=$(get_secret PASSWORD)

if [ -z "$GH_TOKEN" ]; then
  echo "Missing GH_TOKEN Kaggle secret"
  exit 1
fi

if [ -z "$USER_NAME" ] || [ -z "$PASSWORD" ]; then
  echo "Missing USER_NAME or PASSWORD Kaggle secret"
  exit 1
fi

export GH_TOKEN USER_NAME PASSWORD GH_REPO TS_STAMP

log "Finding previous GitHub release"
PREVIOUS_TAG=$(gh api "repos/$GH_REPO/releases?per_page=20" --jq ".[] | select(.tag_name != \"$TS_STAMP\" and .draft == false and .prerelease == false) | .tag_name" | head -n 1)
if [ -z "$PREVIOUS_TAG" ]; then
  printf '  no previous published release found for %s, skipping forum update\n' "$GH_REPO"
  exit 0
fi
printf '  previous release: %s\n' "$PREVIOUS_TAG"

WORK_DIR="$BASE/forum-update"
rm -rf "$WORK_DIR"
mkdir -p "$WORK_DIR/assets" "$WORK_DIR/extracted"
export WORK_DIR PREVIOUS_TAG

log "Downloading previous release assets"
run_step "download_previous_release_assets" gh release download "$PREVIOUS_TAG" --repo "$GH_REPO" --dir "$WORK_DIR/assets" --clobber

log "Extracting previous release archive"
run_step "extract_previous_release" bash -lc '
  shopt -s nullglob
  parts=( "$WORK_DIR"/assets/sefaria-exports-*.tar.zst.part-* )
  archives=( "$WORK_DIR"/assets/sefaria-exports-*.tar.zst )
  if [ "${#parts[@]}" -gt 0 ]; then
    cat "${parts[@]}" > "$WORK_DIR/previous-release.tar.zst"
  elif [ "${#archives[@]}" -gt 0 ]; then
    cp -f "${archives[0]}" "$WORK_DIR/previous-release.tar.zst"
  else
    echo "no archive asset found in previous release"
    exit 1
  fi
  tar --zstd -xf "$WORK_DIR/previous-release.tar.zst" -C "$WORK_DIR/extracted"
'

cat > "$WORK_DIR/publish_otzaria_forum_update.py" <<'PY'
import hashlib
import os
import re
import uuid
from collections import defaultdict
from pathlib import Path

import requests

BASE_URL = "https://otzaria.org/forum"
TOPIC_ID = 20
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:132.0) Gecko/20100101 Firefox/132.0",
    "Accept-Language": "he,he-IL;q=0.8,en-US;q=0.5,en;q=0.3",
}


def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def collect_books(root: Path) -> dict[str, dict[str, str]]:
    books = {}
    for path in sorted(root.rglob("*.txt")):
        if not path.is_file():
            continue
        rel = path.relative_to(root).as_posix()
        if rel.endswith("גירסת ספריה.txt"):
            continue
        books[rel] = {"hash": file_sha256(path), "stem": path.stem}
    return books


def extract_csrf_token(html: str) -> str:
    match = re.search(r'"csrf_token":"([^"]+)"', html)
    if not match:
        raise ValueError("Failed to fetch CSRF token")
    return match.group(1)


class OtzariaForumClient:
    def __init__(self, username: str, password: str):
        self.base_url = BASE_URL
        self.session = requests.Session()
        self.username = username
        self.password = password
        self.csrf_token = None

    def login(self) -> None:
        login_page = self.session.get(f"{self.base_url}/login", headers=HEADERS, timeout=60)
        login_page.raise_for_status()
        self.csrf_token = extract_csrf_token(login_page.text)
        response = self.session.post(
            f"{self.base_url}/login",
            headers={
                **HEADERS,
                "Content-Type": "application/x-www-form-urlencoded; charset=UTF-8",
                "x-csrf-token": self.csrf_token,
            },
            data={
                "username": self.username,
                "password": self.password,
                "_csrf": self.csrf_token,
                "noscript": "false",
                "remember": "on",
            },
            timeout=60,
        )
        response.raise_for_status()

    def send_post(self, content: str, topic_id: int) -> None:
        response = self.session.post(
            f"{self.base_url}/api/v3/topics/{topic_id}",
            json={
                "uuid": str(uuid.uuid4()),
                "tid": topic_id,
                "handle": "",
                "content": content,
                "toPid": None,
            },
            headers={
                **HEADERS,
                "Content-Type": "application/json; charset=utf-8",
                "x-csrf-token": self.csrf_token,
            },
            timeout=60,
        )
        response.raise_for_status()

    def logout(self) -> None:
        if not self.csrf_token:
            return
        self.session.post(
            f"{self.base_url}/logout",
            headers={**HEADERS, "x-csrf-token": self.csrf_token},
            timeout=60,
        )


previous_root = Path(os.environ["PREVIOUS_EXPORTS_DIR"])
current_root = Path(os.environ["CURRENT_EXPORTS_DIR"])
previous_tag = os.environ["PREVIOUS_RELEASE_TAG"]
current_tag = os.environ["CURRENT_RELEASE_TAG"]
username = os.environ["USER_NAME"].strip().replace(" ", "+")
password = os.environ["PASSWORD"].strip()

if not previous_root.exists():
    raise SystemExit(f"Previous exports directory does not exist: {previous_root}")
if not current_root.exists():
    raise SystemExit(f"Current exports directory does not exist: {current_root}")

previous_books = collect_books(previous_root)
current_books = collect_books(current_root)

previous_hashes = defaultdict(list)
for rel_path, data in previous_books.items():
    previous_hashes[data["hash"]].append(rel_path)

new_titles = []
renamed_titles = []
for rel_path, data in current_books.items():
    if rel_path in previous_books:
        continue
    rename_candidates = []
    for old_rel_path in previous_hashes.get(data["hash"], []):
        old_stem = Path(old_rel_path).stem
        if old_stem != data["stem"]:
            rename_candidates.append(old_rel_path)
    if rename_candidates:
        selected_old_rel_path = sorted(rename_candidates)[0]
        renamed_titles.append((Path(selected_old_rel_path).stem, data["stem"]))
    else:
        new_titles.append(data["stem"])

new_titles = sorted(set(new_titles))
renamed_titles = sorted(set(renamed_titles))

if not new_titles and not renamed_titles:
    print("No new or renamed books were found compared to the previous release.")
    raise SystemExit(0)

lines = [
    "# עדכון ספרים חדש",
    "",
    f"השוואה בין השחרור {previous_tag} לשחרור {current_tag}.",
    "",
]
if new_titles:
    lines.append("## נוספו הספרים הבאים")
    lines.extend(f"* {name}" for name in new_titles)
    lines.append("")
if renamed_titles:
    lines.append("## הספרים הבאים קיבלו שם חדש")
    lines.extend(f"* {old_name} -> {new_name}" for old_name, new_name in renamed_titles)
    lines.append("")
content = "\n".join(lines).strip() + "\n"
print(content)

client = OtzariaForumClient(username, password)
try:
    client.login()
    client.send_post(content, TOPIC_ID)
finally:
    client.logout()
PY

export CURRENT_EXPORTS_DIR="$REPO_DIR/exports"
export PREVIOUS_EXPORTS_DIR="$WORK_DIR/extracted"
export CURRENT_RELEASE_TAG="$TS_STAMP"
export PREVIOUS_RELEASE_TAG="$PREVIOUS_TAG"

log "Publishing Otzaria Forum update"
run_step "publish_otzaria_forum_update" python3 "$WORK_DIR/publish_otzaria_forum_update.py"